In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path so `from tools...` imports work from any notebook subfolder.
_p = Path.cwd().resolve()
for _parent in [_p, *_p.parents]:
    if (_parent / 'tools' / 'search_tools.py').exists():
        sys.path.insert(0, str(_parent))
        break
del _p, _parent


In [1]:
%%capture --no-stderr
# %pip install "autogen-agentchat~=0.2.3"

# In Your OAI_CONFIG_LIST file, you must have two configs,
# one with:           "response_format": { "type": "text" }
# and the other with: "response_format": { "type": "json_object" }


[
    {"model": "gpt-4o-mini", "sk-REDACTED": "key go here", "response_format": {"type": "text"}},
]

In [2]:
import autogen
import os
from autogen.agentchat import UserProxyAgent
from autogen.agentchat.assistant_agent import AssistantAgent
from autogen.agentchat.groupchat import GroupChat
os.environ["SERPER_API_KEY"] = "1edefaec0732d11db50b993ba60539510cc55334"
from tools.search_tools import SearchTools




In [3]:
from autogen import ConversableAgent
from autogen import register_function

import os
import json
import requests

def search_internet(query: str) -> str:
        """Useful to search the internet
        about a a given topic and return relevant results"""
        print("Searching the internet...")
        top_result_to_return = 5
        url = "https://google.serper.dev/search"
        payload = json.dumps(
            {"q": query, "num": top_result_to_return, "tbm": "nws"})
        headers = {
            'X-API-KEY': os.environ['SERPER_API_KEY'],
            'content-type': 'application/json'
        }
        response = requests.request("POST", url, headers=headers, data=payload)
        # check if there is an organic key
        if 'organic' not in response.json():
            return "Sorry, I couldn't find anything about that, there could be an error with you serper api key."
        else:
            results = response.json()['organic']
            string = []
            print("Results:", results[:top_result_to_return])
            for result in results[:top_result_to_return]:
                try:
                    # Attempt to extract the date
                    date = result.get('date', 'Date not available')
                    string.append('\n'.join([
                        f"Title: {result['title']}",
                        f"Link: {result['link']}",
                        f"Date: {date}",  # Include the date in the output
                        f"Snippet: {result['snippet']}",
                        "\n-----------------"
                    ]))
                except KeyError:
                    next

            return '\n'.join(string)

        



In [4]:
import asyncio
import autogen
import os
import httpx
from typing import Optional, List, Dict, Tuple, Union
import random  # noqa E402

import matplotlib.pyplot as plt  # noqa E402
import networkx as nx  # noqa E402

import autogen  # noqa E402
from autogen.agentchat.conversable_agent import ConversableAgent  # noqa E402
from autogen.agentchat.assistant_agent import AssistantAgent  # noqa E402
from autogen.agentchat.groupchat import GroupChat  # noqa E402
from autogen.graph_utils import visualize_speaker_transitions_dict 

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "sk-REDACTED"

# Define a custom HTTP client
class MyHttpClient(httpx.Client):
    def __deepcopy__(self, memo):
        return self

# Configure the language model
llm_config = {
    "config_list": [
        {
            "model": "qwen2.5:72b",
            "api_type": "ollama",
            "client_host": "https://dy73p29icm8m1v-11434.proxy.runpod.net",
        }
    ]
}


class MemoryAgent(ConversableAgent):
    def __init__(self, name, system_message, llm_config):
        super().__init__(name=name, system_message=system_message, llm_config=llm_config)
        self.memory = []

    def receive(self, message, sender, request_reply=None, silent=False):
        print(f"[{self.name}] Received message from {sender.name}: {message}")
        if isinstance(message, dict):
            # Check for tool-related keys
            if 'tool_calls' not in message and 'tool' not in message:
                role = message.get('role', '')
                content = message.get('content', '')
                # Store the received message in memory if it's not tool-related
                self.memory.append({"role": role, "content": content})
            else:
                print(f"Info: Tool-related message received and not stored in memory: {message}")
        else:
            print(f"Warning: Received a message without expected structure: {message}")
        # Call the parent class's receive method to handle the message
        super().receive(message, sender, request_reply, silent)

    def send(self, message, recipient, request_reply=None, silent=False):
        print(f"[{self.name}] Sending message to {recipient.name}: {message}")
        if isinstance(message, dict):
            # Check for tool-related keys
            if 'tool_calls' not in message and 'tool' not in message:
                role = message.get('role', '')
                content = message.get('content', '')
                # Store the agent's response in memory if it's not tool-related
                self.memory.append({"role": role, "content": content})
            else:
                print(f"Info: Tool-related message sent and not stored in memory: {message}")
        else:
            print(f"Warning: Sending a message without expected structure: {message}")
        # Call the parent class's send method to send the message
        super().send(message, recipient, request_reply, silent)

    def generate_response(self, messages):
        # Limit to the last 10 messages from memory
        recent_memory = self.memory[-10:]
        # Prepare the conversation context
        conversation = [{"role": "system", "content": self.system_message}] + recent_memory + messages + "choose who do you want to speak next"
        # Generate a response using the language model
        response = self.llm.generate(conversation)
        # Store the response in memory
        self.memory.append({"role": "assistant", "content": response})
        return response


def is_termination_msg(content) -> bool:
    have_content = content.get("content", None) is not None
    if have_content and "TERMINATE" in content["content"]:
        return True
    return False


user_proxy = autogen.UserProxyAgent(
    name="User_proxy",
    system_message="A human admin who terminates the chat when the human user sends a message with 'TERMINATE' mentioned it it",
    code_execution_config=False,
    human_input_mode="ALWAYS",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
)

# Include orders and layout in Leader's instructions
Head_Researcher = MemoryAgent(
    name="Head_Researcher",
    system_message=(
        "You are a head researcher who is experienced in the field of AI, NLP and LLM Agent Frameworks.\n"
        "Help the user in his research work by offering him detailed content for his queries"
        "Use the search function to browse the internet. you have no other function apart from this"
    ),
    llm_config=llm_config,
)





In [5]:
def is_termination_msg(content) -> bool:
    have_content = content.get("content", None) is not None
    if have_content and "TERMINATE" in content["content"]:
        return True
    return False

agents = [Head_Researcher, user_proxy]


group_chat = GroupChat(
    agents=agents,
    messages=[],
    max_round=200,
)


# Create the manager
manager = autogen.GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    code_execution_config=False,
)


In [6]:
register_function(
    search_internet,
    caller=Head_Researcher,  # The assistant agent can suggest calls to the calculator.
    executor=Head_Researcher,  # The user proxy agent can execute the calculator calls.
    name="Search_Internet",  # By default, the function name is used as the tool name.
    description="Browse the internet using this tool",  # A description of the tool
)

In [7]:
# chat_result = user_proxy.initiate_chat(Agent0, message="search internet about google. Use production Manager first")

In [8]:
# Prepare the initial message with the novel's text
def read_moby_dick():
    """Reads the 'flatland.txt' file and returns its content as a string."""
    try:
        with open('flatland.txt', 'r', encoding='utf-8') as file:
            return file.read()
    except FileNotFoundError:
        return "Error: 'flatland.txt' not found in the current directory."
    except Exception as e:
        return f"An error occurred: {e}"
    

# Read the content of 'moby_dick.txt'
flatland_text = read_moby_dick()


initial_message = (
    "you are now going to help the user draft his research paper as per his requirements on LLM agent coordination framework and assist him to your best"
    "Here is the existing research paper as of now - \n\n{flatland_text}"
)



# Initiate the conversation
user_proxy.initiate_chat(manager, message=initial_message)

User_proxy (to chat_manager):

you are now going to help the user draft his research paper on LLM agent coordination framework and assist him to your best

--------------------------------------------------------------------------------


GroupChat is underpopulated with 2 agents. Consider setting speaker_selection_method to 'round_robin' or allow_repeat_speaker to False, or use direct communication, unless repeated speaker is desired.


[Head_Researcher] Received message from chat_manager: {'content': 'you are now going to help the user draft his research paper on LLM agent coordination framework and assist him to your best', 'role': 'user', 'name': 'User_proxy'}

Next speaker: Head_Researcher


>>>>>>>> USING AUTO REPLY...
[Head_Researcher] Sending message to chat_manager: {'content': "Absolutely, I'd be glad to assist you in drafting your research paper on LLM (Large Language Model) Agent Coordination Frameworks. Let’s start by outlining the key sections of the paper:\n\n1. **Introduction**\n   - Background on LLMs and their applications.\n   - Importance of agent coordination in complex tasks.\n   - Overview of existing frameworks.\n\n2. **Literature Review**\n   - Key concepts in multi-agent systems.\n   - Previous work on LLM-based agents.\n   - Comparative analysis of different coordination strategies.\n\n3. **Methodology**\n   - Description of the proposed framework.\n   - Technical details and algorithms used.

GroupChat is underpopulated with 2 agents. Consider setting speaker_selection_method to 'round_robin' or allow_repeat_speaker to False, or use direct communication, unless repeated speaker is desired.



Next speaker: User_proxy

User_proxy (to chat_manager):

terminate

--------------------------------------------------------------------------------


...


Next speaker: Head_Researcher


>>>>>>>> USING AUTO REPLY...
[Head_Researcher] Sending message to chat_manager: {'content': "Certainly! Let's move on to the **Literature Review** section. This section will provide a comprehensive overview of the key concepts in multi-agent systems, previous work on LLM-based agents, and a comparative analysis of different coordination strategies.\n\n### 2. Literature Review\n\n#### Key Concepts in Multi-Agent Systems\n\nMulti-Agent Systems (MAS) are composed of multiple autonomous entities that interact with each other to achieve common or individual goals. The core concepts in MAS include:\n\n- **Agents**: Autonomous entities capable of perceiving their environment and performing actions.\n- **Environment**: The setting in which agents operate, which can be static or dynamic.\n- **Interactions**: Communication and collaboration between agents, often mediated by communication protocols.\n- **Coordination**: Mechanisms to ensure that the actions of mul

GroupChat is underpopulated with 2 agents. Consider setting speaker_selection_method to 'round_robin' or allow_repeat_speaker to False, or use direct communication, unless repeated speaker is desired.



Next speaker: Head_Researcher


>>>>>>>> USING AUTO REPLY...
[Head_Researcher] Sending message to chat_manager: {'content': "Certainly! Let's move on to the **Methodology** section. This section will provide a detailed description of the proposed LLM Agent Coordination Framework (LACF), including its system architecture, components, algorithms, and evaluation metrics.\n\n### 3. Methodology\n\n#### Description of the Proposed LLM Agent Coordination Framework (LACF)\n\nThe LLM Agent Coordination Framework (LACF) is designed to facilitate efficient and effective coordination among multiple large language model (LLM) agents in complex tasks. The framework addresses key challenges such as scalability, adaptability, communication overhead, and conflict resolution by integrating advanced algorithms and protocols.\n\n#### System Architecture and Components\n\nThe LACF consists of several key components that work together to ensure seamless coordination:\n\n1. **Agent Layer**:\n   - **LLM Agen

GroupChat is underpopulated with 2 agents. Consider setting speaker_selection_method to 'round_robin' or allow_repeat_speaker to False, or use direct communication, unless repeated speaker is desired.



Next speaker: User_proxy



In [9]:
last_message = group_chat.messages[-1] if group_chat.messages else None
if last_message:
    print("Final Message Content:", last_message['content'])

Final Message Content: please continue 


In [10]:
def save_conversation_to_file(groupchat, filename="chat.txt"):
    """
    Save the entire conversation history to a specified file.

    Args:
        groupchat (GroupChat): The GroupChat instance containing the messages.
        filename (str): The name of the file to save the conversation history.
    """
    if not groupchat.messages:
        print("No messages in the group chat to save.")
        return

    # Compile the conversation history
    conversation_history = "\n".join(
        f"{msg['role']}: {msg['content']}" for msg in groupchat.messages
    )

    # Write the conversation history to the file
    with open(filename, "w", encoding="utf-8") as file:
        file.write(conversation_history)

    print(f"Conversation history saved to {filename}")
    
save_conversation_to_file(group_chat, filename="chat.txt")


Conversation history saved to chat.txt
